# Taiwan ASR Toolkit — 30-second Quickstart

Transcribe Traditional Chinese (Taiwan Mandarin) audio with **MediaTek Breeze-ASR-25** and **Qwen3-ASR-1.7B**, end-to-end, in ~3 cells.

[GitHub](https://github.com/thc1006/taiwan-asr-toolkit) · [Issues](https://github.com/thc1006/taiwan-asr-toolkit/issues) · MIT License

## What this notebook does

1. Installs the toolkit + a CPU-friendly subset of dependencies.
2. Downloads a bundled 30-second Taiwan-Mandarin sample (no GPU required).
3. Runs Breeze-ASR-25 with hot-word injection.
4. Prints the transcript (Traditional Chinese, with timestamps).

On a Colab T4 GPU runtime the full transcription runs in under 10 seconds. On free-tier CPU it takes ~60 s.

In [ ]:
# Cell 1 — install (Colab + local both supported)
import sys
!{sys.executable} -m pip install -q git+https://github.com/thc1006/taiwan-asr-toolkit.git
# Optional: jiwer (CER eval) and pyannote (diarization)
# !{sys.executable} -m pip install -q jiwer pyannote.audio

In [ ]:
# Cell 2 — fetch the bundled 30-second sample + glossary
import urllib.request, os
BASE = 'https://raw.githubusercontent.com/thc1006/taiwan-asr-toolkit/main'
for name, dst in [
    (f'{BASE}/tests/fixtures/clip_30s.wav', 'clip_30s.wav'),
    (f'{BASE}/glossary.txt',                'glossary.txt'),
]:
    if not os.path.exists(dst):
        urllib.request.urlretrieve(name, dst)
        print('downloaded:', dst)
    else:
        print('cached:', dst)

In [ ]:
# Cell 3 — transcribe with Breeze-ASR-25 + glossary hot-word injection
!asr-breeze clip_30s.wav --glossary-file glossary.txt

In [ ]:
# Cell 4 — read the output (Traditional Chinese with timestamps)
from pathlib import Path
txt = Path('transcripts/breeze/clip_30s_breeze.txt').read_text(encoding='utf-8')
print(txt)

## What you just saw

The audio is a 30-second clip from a Taiwan Mandarin standard recording. Notice that the transcript is in **Traditional Chinese** with **Taiwan idioms** (軟體 not 軟件, 雷射 not 激光) thanks to the OpenCC `s2twp` post-processing baked into the toolkit.

Glossary hot-word injection biases Whisper toward proper nouns at decode time. The included glossary covers NTU dorm and department names; adapt it to your domain by editing `glossary.txt`.

## Next steps

- Try the other model: `!asr-qwen3 clip_30s.wav`
- LLM context-polish a transcript: `!asr-polish transcripts/breeze/clip_30s_breeze.json --glossary-file glossary.txt`
- Speaker diarization (requires HF license accept on pyannote): `!asr-diarize transcripts/breeze/clip_30s_breeze.json clip_30s.wav`
- See [docs/BENCHMARK.md](https://github.com/thc1006/taiwan-asr-toolkit/blob/main/docs/BENCHMARK.md) for benchmark numbers (RTF up to 1554x on RTX 5090).
- See [docs/ARCHITECTURE.md](https://github.com/thc1006/taiwan-asr-toolkit/blob/main/docs/ARCHITECTURE.md) for design rationale.

## License & credits

This toolkit is integration plumbing under MIT. Models retain their own licenses (Apache 2.0 for Qwen and Breeze, gated for pyannote). Please cite the underlying model authors in academic work.